In [1]:
##############
# HOW TO RUN #
##############

# 1. Create the test Dask cluster. This will be used to auto-determine the appropriate block size
#    for your machine! 
from robustraster import dask_cluster_manager

dask_cluster = dask_cluster_manager.DaskClusterManager()
dask_cluster.create_cluster(mode="test")

Dask dashboard is available at: http://127.0.0.1:8787/status


In [1]:
# 2. Load the local raster
from robustraster import input_driver
raster_path_list = ['./ndvi.tif', './ndvi_local.tif']
local_raster = input_driver.RasterDataset(raster_path_list)

In [2]:
print(local_raster.dataset)

<xarray.Dataset> Size: 2MB
Dimensions:      (time: 2, y: 546, x: 459)
Coordinates:
  * time         (time) int64 16B 1 2
  * x            (x) float64 4kB -7.668e+04 -7.665e+04 ... -6.297e+04 -6.294e+04
  * y            (y) float64 4kB 1.886e+05 1.886e+05 ... 2.049e+05 2.05e+05
    spatial_ref  int64 8B 0
Data variables:
    band_1       (time, y, x) float32 2MB 3.386e+03 3.421e+03 ... 2.0 2.0
Attributes:
    AREA_OR_POINT:  Area


In [3]:
# 3. Design your function here! 
 
# My target audience are for users who want to work with
# data frames, so pandas data frames are the only data structures 
# that I support for writing functions. If you'd prefer working 
# with xarrays (or possible other data structures), submit an
# issue and let me know!

# For this demo, we do a basic NDVI calculation.
def compute_ndvi(df):
    # Perform your calculations
    df['ndvi'] = (df['band_1'] + df['band_1']) / (df['band_1'])
    return df

In [5]:
# 5. This code will auto-determine what the best block size
#    should be for your machine. This helps to ensure computations don't 
#    go over resources available and crash your application. Skip this step
#    if want to process the entire dataset as is.
from robustraster import udf_tuner

user_defined_func = udf_tuner.UserDefinedFunction()
user_defined_func.tune_user_function(local_raster, compute_ndvi, None)

None
Difference is GREATER than 1%
Difference is GREATER than 1%
Difference is GREATER than 1%
Difference is GREATER than 1%
Difference is GREATER than 1%
Difference is GREATER than 1%
Difference is GREATER than 1%
Difference is GREATER than 1%
Difference is GREATER than 1%
Difference is GREATER than 1%
Difference is GREATER than 1%
Difference is GREATER than 1%
Difference is GREATER than 1%
Difference is GREATER than 1%
Difference is GREATER than 1%
Difference is GREATER than 1%
Difference is GREATER than 1%
Difference is GREATER than 1%
Optimal chunks written to optimal_chunks_20250127_161517.json


In [6]:
# 6. Shutdown the test cluster and recreate a Dask cluster with
#    full resources needed for the full computation.
dask_client = dask_cluster.get_dask_client
dask_client.shutdown()
dask_cluster.create_cluster(mode="full")

Dask dashboard is available at: http://127.0.0.1:8787/status


In [8]:
# 8. Do the full run and write the results to a geoTIFF!
result = user_defined_func.apply_user_function(local_raster, compute_ndvi)

# If you changed the parameters in step 3, you'll need to play around with the parameters
# below and change them accordingly. Otherwise, it should work as is.
ds_transposed = result.transpose('time', 'y', 'x').rio.write_crs("EPSG:3310")
ds_transposed['ndvi'].isel(time=0).rio.to_raster("ndvi_local.tif")